# MetabTravLR quickstart

Train SpaceTravLR with **harreman metabolite transporter pairs** added as a modulator
group, then read the learned `beta_<export>@<import>` coefficients back out over labeled
gene sets to rank metabolites by effect. We analyze the coefficients directly — **no
perturbation**.

Edit the **Config** and **Gene sets** cells, then run top to bottom. Everything is written
under the dataset directory. On Savio, replace `fit(...)` with the `spawn_worker` cell.

In [1]:
import os, sys
import numpy as np
import pandas as pd
import scanpy as sc

# SpaceTravLR package (src/) + our metab_processing helpers
_here = os.path.dirname(os.path.abspath('.'))
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
sys.path.append(os.path.join(os.getcwd(), '..'))

from SpaceTravLR.spaceship import SpaceShip
from metab_processing.metab_loader import load_metab_pairs
from metab_processing import beta_analysis

In [ ]:
# from harreman_summary import select_tcell_metabolites
# select_tcell_metabolites(f'{dataset_dir}/easy_download')

Wrote 76 metabolites -> /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Primary_Dermal_Melanoma/easy_download/harreman_outputs/metabolite_selection.yaml


PosixPath('/global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Primary_Dermal_Melanoma/easy_download/harreman_outputs/metabolite_selection.yaml')

## Config — data dir / dataset selection

Layout assumed: `DATA_DIR / DATASET / {adata.h5ad, easy_download/harreman_outputs/...}`.
Results are written to `DATA_DIR / DATASET / spacetravlr_output`.

In [2]:
DATA_DIR = '/global/scratch/users/fosterangus/MetabTravLR/Data/Xenium'
DATASET  = 'Primary_Dermal_Melanoma'   # dataset folder under DATA_DIR

CELL_TYPE_SRC = 'leiden_scVI_res_0.5'   # adata.obs column to use as 'cell_type' (the harreman tier annotation)

dataset_dir    = f'{DATA_DIR}/{DATASET}'
adata_path     = f'{dataset_dir}/adata.h5ad'
harreman_dir   = f'{dataset_dir}/easy_download/harreman_outputs'
selection_yaml = f'{harreman_dir}/metabolite_selection.yaml'
outdir         = f'{dataset_dir}/spacetravlr_output'
betadata_dir   = f'{outdir}/betadata'

for p in (adata_path, selection_yaml):
    assert os.path.exists(p), f'missing: {p}'

## Gene sets

The target genes to train and the labels to score metabolites against. `focus_genes` (the
genes actually trained) is the union of all sets. **Edit these lists.** With exactly
`positive`/`negative` labels, the ranking uses `signed = positive − negative`.

In [3]:
GENE_SETS = {
    'positive': ['CD4', 'CD3E', 'IL2RA'],          # e.g. T-cell activity
    'negative': ['CTLA4', 'FOXP3', 'IL10', 'ENTPD1'],  # e.g. exhaustion
}

focus_genes = list(dict.fromkeys(g for genes in GENE_SETS.values() for g in genes))
print(f'{len(focus_genes)} focus genes:', focus_genes)

7 focus genes: ['CD4', 'CD3E', 'IL2RA', 'CTLA4', 'FOXP3', 'IL10', 'ENTPD1']


In [4]:
adata = sc.read_h5ad(adata_path)
adata.obs['cell_type'] = adata.obs[CELL_TYPE_SRC]
adata.layers['raw_count'] = adata.X
adata

AnnData object with n_obs × n_vars = 112551 × 5006
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', '_scvi_batch', '_scvi_labels', 'leiden_scVI_res_0.5', 'leiden_scVI_res_2.5', 'leiden_scVI_res_2', 'leiden_scVI_res_1.5', 'leiden_scVI_res_1', 'leiden_scVI_res_0.75', 'leiden_scVI_res_0.65', 'leiden_scVI_res_0.375', 'leiden_scVI_res_0.25', 'leiden_scVI_res_0.1', 'leiden_scVI_res_0.05', 'cd8', 'cd4', 't_cell', 'Tier1', 'Tier2', 'Tier3', 'Cytotoxic_CD8_score', 'Exhausted_CD8_score', 'Treg_score', 'sub_cluster_5_res_1', 'sub_cluster_5_res_0.75', 'sub_cluster_5_res_0.5', 'sub_cluster_5_res_0.37', 'sub_cluster_5_res_0.25', 'sub_cluster_5_res_0.15', 'sub_cluster_5_res_0.1', 'sub_cluster_5_res_0.05', 'sub_cluster_2_res_1', 'sub_cluster_2_res_0.75', 'sub_cluster_2_res_0

## Metabolite pairs from harreman

`metabolite_selection.yaml` → the deduped `metab_pairs` list (homotypic once, heterotypic
both orientations) filtered to genes in the panel. `selection` keeps the metabolite→pairs
grouping for the read-back.

In [5]:
metab_pairs, selection = load_metab_pairs(selection_yaml, var_names=adata.var_names)
print(f'{len(selection)} metabolites, {len(metab_pairs)} model pairs (both orientations, in-panel)')
metab_pairs[:8]

build_metab_pairs: dropped 0 of 144 metab_pairs (gene absent from var_names); kept 144
76 metabolites, 144 model pairs (both orientations, in-panel)


[('ABCB1', 'ABCB1'),
 ('SLC15A1', 'SLC15A1'),
 ('ABCA1', 'ABCA1'),
 ('ATP7A', 'ATP7A'),
 ('ATP7A', 'ATP7B'),
 ('ATP7B', 'ATP7A'),
 ('SLC16A4', 'SLCO2B1'),
 ('SLCO2B1', 'SLC16A4')]

## Setup + train

COMMOT is skipped — harreman is our metabolite prior. Only `focus_genes` are trained.

In [6]:
spacetravlr = SpaceShip(
    name=DATASET.replace('/', '_'),
    outdir=outdir,
    genes=focus_genes,
)

In [7]:
spacetravlr.setup_(adata, overwrite=False, run_commot=False)
assert spacetravlr.is_everything_ok()

AssertionError: Launch script not found

In [8]:
# Local / single-process training. On Savio use the spawn_worker cell below instead.
spacetravlr.fit(metab_pairs=metab_pairs)

Fitting CTLA4 with 960 modulators
	33 Transcription Factors
	637 Ligand-Receptor Pairs
	146 TranscriptionFactor-Ligand Pairs
	0 Extra modulators
	144 Metabolite Pairs
0: 0.2852 | 0.2306
1: 0.5157 | 0.4747
2: 0.3621 | 0.3573
3: 0.5293 | 0.4368
4: 0.3331 | 0.3082
5: 0.2746 | 0.2186
6: 0.2139 | 0.2138
7: 0.3708 | 0.3305
8: 0.6786 | 0.6769
9: 0.2175 | 0.1854
10: 0.4326 | 0.4321
Deleted lock for CD3E after 3600 seconds
Fitting CD4 with 1161 modulators
	67 Transcription Factors
	639 Ligand-Receptor Pairs
	311 TranscriptionFactor-Ligand Pairs
	0 Extra modulators
	144 Metabolite Pairs
0: 0.6285 | 0.6106
1: 0.8061 | 0.7749
2: 0.7035 | 0.7066
3: 0.9225 | 0.9109
4: 0.6649 | 0.6258
5: 0.8935 | 0.8681
6: 0.4683 | 0.4449
7: 0.7344 | 0.6936
8: 0.6847 | 0.6505
9: 0.6613 | 0.6012
10: 0.6679 | 0.6008
Fitting IL10 with 1268 modulators
	85 Transcription Factors
	638 Ligand-Receptor Pairs
	401 TranscriptionFactor-Ligand Pairs
	0 Extra modulators
	144 Metabolite Pairs
0: x.xxxx | 0.1629
1: 0.3625 | 0.3325
2

KeyboardInterrupt: 

In [ ]:
# --- Savio: run this cell (multiple times) to spawn parallel SLURM workers instead of fit() ---
# spacetravlr.focus_genes = focus_genes
# spacetravlr.spawn_worker(
#     account='fc_wagnerlabfca',
#     partition='savio4_gpu',
#     qos='a5k_gpu4_normal',
#     gres='gpu:A5000:1',
#     job_name='MetabTravLR',
#     cpus_per_task=4,
#     lifespan=0.5,
#     python_path='/global/home/users/fosterangus/.conda/envs/spacetravlr/bin/python',
#     metab_pairs=metab_pairs,   # if driving via a launch.py, pass metab_pairs to run_spacetravlr
# )

## Read the metabolite coefficients back out

Per-`(gene, pair, cell_type)` beta summary → roll up to metabolites (optionally weighting
each transporter pair by its harreman `C_np` communication score) → signed gene-set ranking.

In [ ]:
# Per-(gene, export, import, cell_type) mean/std of the learned metabolite beta.
pair_summary = beta_analysis.read_metab_beta_summary(
    betadata_dir,
    genes=focus_genes,
    obs=adata.obs,
    cell_type_col='cell_type',
)
pair_summary.head()

In [ ]:
# Optional: weight transporter pairs by harreman C_np (real-communication score).
# Set weights=None for a plain mean across a metabolite's pairs.
try:
    weights = beta_analysis.gene_pair_cnp_weights(harreman_dir, agg='max')
except Exception as e:
    print(f'C_np weights unavailable ({e}); falling back to unweighted mean')
    weights = None

metab_summary = beta_analysis.aggregate_to_metabolite(pair_summary, selection, weights=weights)
metab_summary.head()

In [ ]:
# Signed metabolite ranking: mean over 'positive' genes − mean over 'negative' genes.
ranking = beta_analysis.gene_set_score(metab_summary, GENE_SETS)
ranking.head(20)